In [1]:
import os
import time
import platform
from shutil import rmtree
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as image
import seaborn as sns

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid

# 1. 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':  # Mac OS
    plt.rc('font', family='AppleGothic')
else:  # Linux (Colab 등)
    plt.rc('font', family='NanumBarunGothic')

# 2. 마이너스 기호(-) 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 오류 발생시
sklearn.set_config(display="text")

# CNN을 활용해서 도형 분류하기

In [2]:
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.Grayscale(1),
    transforms.ToTensor(),
    transforms.Normalize(0.5, 0.5),
])

test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.Grayscale(1),
    transforms.ToTensor(),
    transforms.Normalize(0.5, 0.5),
])

In [3]:
base_path = r"C:\kdt_0900_hhj\ml\resource\dataset"
train_path = os.path.join(base_path, "shape", "train")
test_path = os.path.join(base_path, "shape", "test")

In [4]:
# 데이터 (이미지) 불러오기
train_dataset = ImageFolder(train_path, transform=train_transform)
test_dataset = ImageFolder(test_path, transform=test_transform)

print(len(train_dataset), len(test_dataset))

240 60


In [5]:
class_dict = dict(enumerate(train_dataset.classes))
class_dict

{0: 'cir', 1: 'tri', 2: 'x'}

In [6]:
# DataLoader 만들기
# 이미지 확인하기
# CNN 모델 만들기
# 모델 학습하기
# 테스트 데이터로 정확도 확인하기

### DataLoader

In [7]:
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [8]:
print(len(train_dataset), len(test_dataset))
print(len(train_dataloader), len(test_dataloader))

240 60
4 1


In [9]:
img, label = train_dataset[0]

print(img.shape)
print(label)

torch.Size([1, 64, 64])
0


### 이미지 확인하기

In [ ]:
imgs, labels = next(iter(train_dataloader))

In [ ]:
print(len(imgs), len(labels))

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(5, 5))

for i, ax in enumerate(axes.flatten()):
    ax.imshow(imgs[i].squeeze(), cmap="gray")
    ax.set_title(train_dataset.classes[labels[i].item()])
    ax.axis("off")

plt.show()

In [ ]:
# 쌤 코드
# fig, axes = plt.subplot(8, 8, figsize=(16, 16))

# for ax, img label in zip(axes.flatten(), imgs, labels):
#     ax.imshow(img.reshape(64, 64), cmap="gray")
#     ax.set_title(class_dict.get(label.item(), 0))
#     ax.axis("off")

# plt.show()

### CNN 모델 만들기

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")
elif torch.xpu.is_available():
    device = torch.device("xpu")
else:
    device = torch.device("cpu")

print(f"현재 선택된 장치: {device}")

In [ ]:
class ConvNeuralNetwork(nn.Module):
    def __init__(self):
        super(ConvNeuralNetwork, self).__init__()
        
        self.flatten = nn.Flatten()
        
        self.classifier = nn.Sequential(
            nn.Conv2d(1, 28, kernel_size=3, padding="same"), #64, 64
            nn.ReLU(), # 활성화 함수(선형 -> 비선형)
            
            nn.Conv2d(28, 28, kernel_size=3, padding="same"),
            nn.ReLU(),
            
            nn.MaxPool2d(kernel_size=2), # 32, 32
            nn.Dropout(0.25),
            
            nn.Conv2d(28, 56, kernel_size=3, padding="same"),
            nn.ReLU(),
            
            nn.Conv2d(56, 56, kernel_size=3, padding="same"),
            nn.ReLU(),
            
            nn.MaxPool2d(kernel_size=2), # 16, 16
            nn.Dropout(0.25)
        )
        self.linear = nn.Linear(56 * 16 * 16, 3)
    
    def forward(self, x):
        x = self.classifier(x)
        x = self.flatten(x)
        x = self.linear(x)
        return x

In [ ]:
model = ConvNeuralNetwork().to(device)

In [ ]:
model

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, log_step=20):
    model.train()
    total = 0
    running_loss = 0.0
    running_corrects = 0

    for i, (imgs, labels) in enumerate(dataloader, 1):
        imgs, labels = imgs.to(device), labels.to(device)

        # 초기화
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = imgs.size(0)
        running_loss += loss.item() * batch_size
        _, preds = outputs.max(1)
        running_corrects += (preds == labels).sum().item()
        total += batch_size

        if log_step and i % log_step == 0:
            batch_acc = (preds == labels).float().mean().item()
            print(f"[Train] batch {i}/{len(dataloader)} | Loss: {loss:.4f} | Accuracy: {batch_acc:.2f}")

    return running_loss / total, running_corrects / total

In [ ]:
# model: 학습할 CNN 모델
# dataloader: 학습 이미지를 묶음(batch) 단위로 제공
# criterion: 예측이 얼마나 틀렸는지 계산
# optimizer: 모델의 가중치를 업데이트
# device: CPU 또는 GPU
# log_step=20: 20개 배치마다 중간 결과 출력

In [ ]:
def evaluate(model, dataloader, criterion, device):
    model.train()

    total = 0
    running_loss = 0.0
    running_corrects = 0

    with torch.no_grad():
        for i, (imgs, labels) in enumerate(dataloader, 1):
            imgs, labels = imgs.to(device), labels.to(device)
    
            # 초기화
            outputs = model(imgs)
            loss = criterion(outputs, labels)
    
            batch_size = imgs.size(0)
            running_loss += loss.item() * batch_size
            _, preds = outputs.max(1)
            running_corrects += (preds == labels).sum().item()
            total += batch_size
    

    return running_loss / total, running_corrects / total

### 학습

In [ ]:
epochs = 10

for i in range(epochs):
    # 학습
    train_loss, train_acc = train_one_epoch(
        model=model,
        dataloader=train_dataloader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    # 평가
    test_loss, test_acc = evaluate(
        model=model,
        dataloader=test_dataloader,
        criterion=criterion,
        device=device
    )

    print(f"Epoch: {i}/{epochs}")
    print(f"[Train] Loss: {train_loss:.4f} | Accuracy: {train_acc:.4f}")
    print(f"[Test] Loss: {test_loss:.4f} | Accuracy: {test_acc:.4f}")

### 평가

In [ ]:
model.eval()

imgs, labels = next(iter(test_dataloader))
imgs_dev = imgs.to(device)

with torch.no_grad():
    outputs = model(imgs_dev)
    _, preds = outputs.max(1)

img_list_cpu = imgs.cpu()
y_pred_list_cpu = preds.cpu()
y_true_list_cpu = labels.cpu()

# 시각화
fig, axes = plt.subplots(6, 10, figsize=(16, 8))

for ax, img, y_pred, y_true in zip(axes.flatten(), img_list_cpu, y_pred_list_cpu, y_true_list_cpu):
    ax.imshow(img.squeeze(), cmap="gray")
    # ax.imshow(img.permute(1, 2, 0), cmap="gray")

    pred_label = class_dict[y_pred.item()]
    true_label = class_dict[y_true.item()]
    color = "green" if pred_label == true_label else "red"
    ax.axis("off")
    ax.set_title(f"Pred: {pred_label}\nTrue: {true_label}", color=color)

plt.show()

In [ ]:
print(img_list_cpu.shape)
print(len(y_pred_list_cpu))
print(len(y_true_list_cpu))

In [ ]:
# # 쌤 코드

# img_path = ""

# shape = Image.open(img_path).convert("RGB")
# input_tensor = test_transform(shape)
# input_tensor_dev = input_tensor.unsqueeze(0).to(device)
# # 
# with torch.no_grad():
#     outputs = model(input_tensor_dev)
#     probs = F.softmax(outputs, dim=1)[0]

#     pred_idx = torch.argmax(probs).item()
#     pred_class = class_dict[pred_idx]
#     confidence = probs[pred_idx].item() * 100

#     print(f"예측 결과: {pred_class}, 확률: {confidence}%")

-------------------------------------------------------------------------------------------------------------------------

## Grad-CAM(Gradient-weighted Class Activation Mapping)
- 딥러닝 인공지능(AI) 모델이 이미지를 보고 어떤 영역을 근거로 특정 클래스(결과)를 예측했는지 시각화해 주는 설명 가능한 AI(XAI, Explainable AI) 기법입니다

In [ ]:
# 훅(Hook)을 사용하여 마지막 Convolution 층의 활성화 값과 그래디언트 추출 준비
activations = None
gradients = None

def forward_hook(module, input, output):
  global activations
  activations = output

def backward_hook(module, grad_input, grad_output):
  global gradients
  gradients = grad_output[0]


# 우리 모델의 classifier(또는 features) 안에서 마지막 Conv 레이어 지정
# 모델 구조에 맞게 마지막 Conv2d 레이어를 타겟으로 잡기
target_layer = model.classifier[-5]  # 마지막 직전 Conv2d 또는 마지막 Conv2d
target_layer.register_forward_hook(forward_hook)
target_layer.register_backward_hook(backward_hook)

# 테스트할 이미지 경로 설정 및 전처리 적용
original_image = Image.open(img_path).convert("RGB")

# test_transform 활용
input_tensor = test_transform(original_image).unsqueeze(0).to(
    device
)  # (C, H, W) -> (1, C, H, W)

# 순전파 및 추론
output = model(input_tensor)
probs = F.softmax(output, dim=1)[0]
pred_idx = torch.argmax(probs).item()
pred_class = class_dict[pred_idx]  # 클래스 이름 딕셔너리 (예: {0: 'circle', ...})
confidence = probs[pred_idx].item() * 100

print(f"예측 결과: {pred_class} (확률: {confidence:.1f}%, 클래스 번호: {pred_idx})")

# Grad-CAM 역전파 및 히트맵 계산
model.zero_grad()
class_score = output[0, pred_idx]
class_score.backward()

# 채널별 중요도 가중치 계산 (GAP 성향 반영)
pooled_gradients = torch.mean(gradients, dim=[0, 2, 3])
for i in range(activations.size(1)):
    activations[:, i, :, :] *= pooled_gradients[i]

heatmap = torch.mean(activations, dim=1).squeeze().cpu().detach().numpy()
heatmap = np.maximum(heatmap, 0)  # ReLU 적용 (양수 기여도만)
if heatmap.max() != 0:
    heatmap /= heatmap.max()  # 0~1 정규화

# PIL을 이용해 히트맵 크기를 원본 이미지 크기에 맞게 조절
heatmap_uint8 = np.uint8(255 * heatmap)
heatmap_pil = Image.fromarray(heatmap_uint8).resize(
    original_image.size, Image.Resampling.BILINEAR
)

# Matplotlib으로 원본 이미지와 Grad-CAM 오버레이 시각화
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# 첫 번째: 원본 이미지 및 예측 결과
axes[0].imshow(original_image)
axes[0].set_title(f"My Input Image\nPred: {pred_class} ({confidence:.1f}%)", fontsize=11, color="blue",)
axes[0].axis("off")

# 두 번째: 원본 위에 히트맵 겹치기 (alpha=0.4로 반투명 효과)
axes[1].imshow(original_image)
axes[1].imshow(heatmap_pil, cmap="jet", alpha=0.4)  # jet 컬러맵으로 무지개빛 히트맵 표현
axes[1].set_title(f"Grad-CAM Heatmap\n(Where model focused)", fontsize=11, color="red")
axes[1].axis("off")

plt.tight_layout()
plt.show()